# Modes at stopband edges: the fixed edge basis

Dispersion engineering puts stopbands where some of the modes of a mode array land. As the signal is swept, a mode without input (an idler, a conversion product, a pump harmonic) can cross a **stopband edge**. There the two Bloch modes of the supercell coalesce: the transfer matrix becomes a Jordan block (an exceptional point), the normalisation $W = Z_b^+ + Z_b^-$ vanishes, and every coupling of that mode, proportional to $1/W$, diverges. The forward–backward solver integrates the two directions in separate sweeps, which breaks the cancellation between the large forward and backward amplitudes, and the iteration fails.

The forward–backward Bloch model (`model="bloch"`, uniform devices) handles these modes with a **fixed edge basis**, set by `FBSolverConfig.edge_basis_threshold` (default 0.10, `None` disables it). A mode without input whose splitting $\lvert\sinh\gamma d\rvert$ is below the threshold is kept, expanded on a fixed Bloch pair, and its Bragg coupling is solved exactly.

This notebook compares the gain across the **whole gain band** of the V3 holotype with the edge basis on and off, and shows where it matters.

In [ ]:
import logging

import edge_basis_sweep as E
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from edge_basis_diagram import draw
from palette import PALETTE

from twpasolver.logger import log

log.setLevel(logging.ERROR)
SECTOR_COLORS = [
    (PALETTE["purple"], PALETTE["purple_light"]),
    (PALETTE["gold"], PALETTE["gold_light"]),
    (PALETTE["magenta"], PALETTE["magenta_light"]),
]
INK, GREY = PALETTE["ink"], PALETTE["grey"]

## How the edge basis works

1. **(a) Native basis.** Each mode is expanded on the two eigenvectors $\psi^\pm$ of the supercell transfer matrix at its own frequency. Towards an edge they close up, and at the edge they become a single standing wave. $W$, the "area" they span, goes to zero, and the projected couplings $\propto 1/W$ diverge.
2. **(b) Fixed basis.** A flagged mode is expanded instead on the lossless Bloch pair $\tilde\psi^\pm$ at the reference frequency $f_\mathrm{ref}$ on the passband side of the edge where the splitting equals the threshold, $e = 0.10$. The basis is anchored at the switch-over point, so on the passband side the fixed and native bases coincide where the mode enters the window. Any fixed basis is exact. The true eigenvectors are combinations $v^\pm, w^\pm$ of the fixed pair, and $W = \tilde W\,(v_+w_- - v_-w_+)$ vanishes only through those coefficients, which the solver never computes. $\tilde W$ is fixed, real and finite.
3. **(c) Along the line.** Because the fixed basis does not diagonalise the supercell, one supercell mixes the two amplitudes: $dc/dx = \tilde K c + S$, with $\tilde K = \log[(-1)^m P]/d$ a constant $2\times2$ matrix. Its off-diagonal entries are the **Bragg coupling** between forward and backward waves. Its eigenvalues are imaginary in the passband, zero at the edge and real in the gap. This is the coupled-wave theory of Bragg gratings, and it is regular at the edge because nothing is diagonalised.
4. **(d) Solver.** The Bragg coupling of a flagged mode is far too strong to split between a forward and a backward sweep ($|\tilde K_{12}| L$ reaches 50–90). Each pass therefore predicts $c^+$ in the forward sweep, freezes $c^-$ in the backward sweep, and then solves the $2\times2$ two-point problem exactly with a Riccati sweep. The extra cost is a few percent per pass.

**Why 0.10 and not smaller.** The threshold sets two opposite cell-averaging errors. In the fixed basis the nonlinear source carries $1/\tilde W$, and $\tilde W$ is the native $W$ at the anchor, which scales like $e$: anchoring close to the edge brings back part of the $1/W$ problem. The Bragg term of $\tilde K$ instead grows like $e$. Around 0.1 both are small. On the holotype the gain at the edge changes by 0.005–0.015 dB between 0.10 and 0.12, but by about 0.13 dB between 0.06 and 0.12.

The nonlinear couplings keep their strength. The current through every nonlinear inductor is the same in both bases, and the slow-light enhancement that the native basis puts in $1/W$ now comes from the response of $\tilde K$: the standing wave that a slow mode builds up.

In [ ]:
fig = draw()

## Which modes come close to an edge

We use the thesis operating point: pump at 15.8 GHz, loss tangent $2\times10^{-4}$, and the 9-mode array of the thesis, organised in the three sectors of the mode graph:

- **minimal** $(p, s, i)$;
- **1st FC** $(p_2, ps, pi)$;
- **2nd FC** $(p_3, p_2s, p_2i)$.

The splitting $e = |\sinh\gamma d|$ of each signal-dependent mode is computed directly from the supercell matrix. It vanishes at every band edge, and the edge basis takes over below $0.10$ (dotted line). The pump and its harmonics never come close to an edge at this pump frequency. The signal itself does, near 13.4 and 14.3 GHz, but it carries the input and keeps its native basis; the solver only warns.

In [ ]:
an = E.make_analysis()
fs_dense = np.arange(*E.FS_RANGE, 1e-3)
split = E.mode_splitting(an.twpa, fs_dense)

fig, ax = plt.subplots(figsize=(7, 2.8))
style = {"s": "-", "ps": "-", "p2s": "-", "i": "--", "pi": "--", "p2i": "--"}
for m in E.WEAK:
    ax.semilogy(
        fs_dense,
        split[m],
        style[m],
        color=SECTOR_COLORS[E.SECTOR_OF[m]][0],
        lw=1.0,
        label=m,
    )
ax.axhline(E.E_WINDOW, color=GREY, lw=0.8, ls=":")
ax.text(
    E.FS_RANGE[0] + 0.05,
    E.E_WINDOW * 1.08,
    "edge basis",
    color=GREY,
    ha="left",
    va="bottom",
    fontsize=7,
)
ax.set_ylim(5e-3, 3)
ax.set_xlim(*E.FS_RANGE)
ax.set_xlabel("Signal frequency (GHz)")
ax.set_ylabel(r"$|\sinh\gamma d|$")
ax.legend(ncol=6, fontsize=7, loc="upper center", frameon=False)
ax.set_title(
    "Splitting of the signal-dependent modes (solid: s branch, dashed: i branch)",
    fontsize=8,
)
plt.show()

## Full-band gain with and without the edge basis

The sweep runs the forward–backward Bloch CMEs (`model="bloch"`, 60 passes, Anderson depth 30, 40 Ω ports) on a grid that is coarse across the band (50 MHz) and fine inside every near-edge window (4 MHz). It does this twice:

- **edge basis off** (`edge_basis_threshold=None`): native Bloch basis everywhere;
- **edge basis on** (default): modes without input below the threshold in the fixed basis.

Each run takes about a minute on a multicore machine. The results are also cached in `edge_basis_data/full_band.npz`; call `E.run_sweep()` without `recompute=True` to load them.

In [ ]:
d = E.run_sweep(recompute=True)
fs = d["fs"]
wins = d["windows"]
G = {k: d[k]["gain_db"] for k in ("none", "edge")}
labels = d["edge"]["labels"]


def failed(run):
    # Not converged, or non-finite (an overflow makes the residual read as zero).
    return ~d[run]["converged"] | ~np.isfinite(d[run]["gain_db"])


fail = {k: failed(k) for k in G}
flag_any = d["edge"]["edge_flag"].any(axis=1)
print(f"{len(fs)} signal points; edge basis active at {flag_any.sum()}")
print("failed points:", {k: int(v.sum()) for k, v in fail.items()})

In [ ]:
def shade_windows(ax, label=True, ymax=None):
    for m, ws in wins.items():
        dark, light = SECTOR_COLORS[E.SECTOR_OF[m]]
        for a, b, _ in ws:
            if m == "s":  # injected signal: never flagged
                ax.axvspan(a, b, fc="none", ec=GREY, hatch="////", lw=0, zorder=0)
            else:
                ax.axvspan(a, b, color=light, lw=0, zorder=0)
            if label:
                ax.text(
                    0.5 * (a + b),
                    ymax,
                    m,
                    color=dark if m != "s" else GREY,
                    ha="center",
                    va="bottom",
                    fontsize=7,
                )


fig, (ax0, ax1) = plt.subplots(
    2,
    1,
    figsize=(7, 4.6),
    sharex=True,
    gridspec_kw={"height_ratios": [2.2, 1], "hspace": 0.08},
)
ok = ~fail["none"]
ax0.plot(
    fs[ok],
    G["none"][ok],
    ".",
    color=PALETTE["blue"],
    ms=2.5,
    label="edge basis off (converged points)",
)
ax0.plot(fs, G["edge"], color=INK, lw=0.9, label="edge basis on")
ymin, ymax = -8, 30
ax0.plot(
    fs[fail["none"]],
    np.full(fail["none"].sum(), ymin + 1),
    "x",
    color=PALETTE["coral"],
    ms=4,
    label="edge basis off: failed",
)
ax0.set_ylim(ymin, ymax)
shade_windows(ax0, ymax=ymax)
ax0.annotate(
    "signal in its\nown stopband",
    xy=(13.9, ymin + 0.5),
    xytext=(12.6, 4),
    fontsize=7,
    color=GREY,
    arrowprops=dict(arrowstyle="-", color=GREY, lw=0.6),
)
ax0.set_ylabel("Signal gain (dB)")
ax0.legend(fontsize=7, loc="upper left", bbox_to_anchor=(0.0, 0.93), frameon=False)

ax1.axhline(0, color=GREY, lw=0.6)
ax1.plot(fs[ok], (G["edge"] - G["none"])[ok], ".", color=INK, ms=2.5)
shade_windows(ax1, label=False)
ax1.set_ylabel(r"$G_\mathrm{on}-G_\mathrm{off}$ (dB)")
ax1.set_xlabel("Signal frequency (GHz)")
ax1.set_xlim(*E.FS_RANGE)
plt.show()

Shaded bands are the windows where the edge basis is active (splitting below 0.10) for each signal-dependent mode, coloured by its sector (minimal purple, 1st FC gold, 2nd FC magenta). The hatched bands are the windows of the signal itself, which carries the input and keeps its native basis. Between them (13.5–14.2 GHz) the signal is inside the first stopband and is not transmitted, which is physical. Outside the windows both runs solve the same equations and coincide exactly; the lower panel shows the difference where the run without the edge basis converged. Most windows are 100–200 MHz wide, so the zooms below show what happens inside each one.

In [ ]:
flagged_windows = [(m, a, b, e) for m in E.WEAK if m != "s" for a, b, e in wins[m]]
flagged_windows.sort(key=lambda w: w[1])
n = len(flagged_windows)
ncol = 5
fig, axs = plt.subplots(
    int(np.ceil(n / ncol)), ncol, figsize=(7.2, 1.9 * np.ceil(n / ncol)), squeeze=False
)
for ax, (m, a, b, e) in zip(axs.flat, flagged_windows):
    dark, light = SECTOR_COLORS[E.SECTOR_OF[m]]
    sel = (fs >= a - E.MARGIN - 1e-9) & (fs <= b + E.MARGIN + 1e-9)
    ax.axvspan(a, b, color=light, lw=0)
    okn = sel & ~fail["none"]
    ax.plot(fs[okn], G["none"][okn], ".", color=PALETTE["blue"], ms=3)
    ax.plot(fs[sel], G["edge"][sel], color=INK, lw=1.0)
    lo, hi = np.nanmin(G["edge"][sel]), np.nanmax(G["edge"][sel])
    pad = max(0.3, 0.15 * (hi - lo))
    ax.set_ylim(lo - pad, hi + pad)
    bad = sel & fail["none"]
    ax.plot(
        fs[bad], np.full(bad.sum(), lo - 0.6 * pad), "x", color=PALETTE["coral"], ms=3.5
    )
    ax.set_title(
        f"{m} ({E.SECTOR_NAMES[E.SECTOR_OF[m]]})", color=dark, fontsize=8, pad=2
    )
    ax.tick_params(labelsize=6.5)
    ax.locator_params(axis="x", nbins=3)
for ax in axs.flat[n:]:
    ax.axis("off")
fig.supxlabel("Signal frequency (GHz)", fontsize=8)
fig.supylabel("Signal gain (dB)", fontsize=8)
fig.tight_layout()
plt.show()

In each zoom: solid black is the gain with the edge basis, blue dots the gain without it where that run converged, and red crosses the points where it failed. The shaded band is the edge-basis window of the mode in the title. The fast zigzag is the Fabry–Pérot ripple of the 40 Ω ports, about ±4 dB with a period close to 8 MHz, which the 4 MHz grid aliases. Lower `E.FINE_STEP` to resolve it (at a proportional cost).

In [ ]:
rows = []
for m, a, b, e in flagged_windows:
    sel = (fs >= a - 1e-9) & (fs <= b + 1e-9)
    j = labels.index(m)
    both = sel & ~fail["none"]
    rows.append(
        {
            "mode": m,
            "sector": E.SECTOR_NAMES[E.SECTOR_OF[m]],
            "window (GHz)": f"{a:.3f}-{b:.3f}",
            "min split": round(float(e), 3),
            "points": int(sel.sum()),
            "edge basis on": int(d["edge"]["edge_flag"][sel, j].sum()),
            "off: failed": int(fail["none"][sel].sum()),
            "max |G_on - G_off| (dB)": (
                round(float(np.nanmax(np.abs(G["edge"][both] - G["none"][both]))), 3)
                if both.any()
                else None
            ),
            "on: converged": f"{int(d['edge']['converged'][sel].sum())}/{int(sel.sum())}",
        }
    )
pd.DataFrame(rows)

## What to take away

- **Where it matters.** Every signal-dependent mode crosses band edges somewhere in the band, not only $p_2s$. Without the edge basis the iteration fails at 21 points, all inside these windows (red crosses). With it, every point in the windows converges; the only failure left is at 14.313 GHz, where the signal itself sits at its own stopband edge.
- **What the edge basis changes.** Outside the windows the two runs are identical. Inside, where the native run still converges, they differ by up to 0.3–0.55 dB in the $pi$ and $ps$ windows (1st FC), about 0.1 dB for $p_2i$ and $p_2s$ (2nd FC), and 0.02 dB for the idler $i$, whose windows lie where there is no gain.
- **Converged is not the same as right.** Inside a window the native run sometimes converges, but its couplings carry the $1/W$ factor and the cell averaging is breaking down, so these points are not a reference. The checks of the edge basis are its agreement with the native basis where both are valid, the seamless switch on the passband side, and its weak dependence on the threshold around 0.1 (see `tests/test_bloch_edge_basis.py`).
- **Cost.** The edge basis adds a few percent per pass at flagged points and nothing elsewhere.
- **Convergence.** Near an edge it needs more passes than elsewhere. With the solver defaults (30 passes, Anderson depth 5) only about half of the points near the $p_2s$ edge of the reference device converge, and the gain differs by up to 0.08 dB; this notebook uses 60 passes and depth 30.

The edge basis acts on every mode without input, pump harmonics included. A mode with an input close to an edge, such as the signal at 13.4 and 14.3 GHz here, keeps its native basis and is only reported.